# APS Model Selection 




In [1]:
from pathlib import Path
import json
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    import tensorflow as tf
except Exception:
    tf = None

seed = 42
target = 'class'

root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent

train_path = root / 'data' / 'processed' / 'aps_train_cleaned.csv'
test_path = root / 'data' / 'processed' / 'aps_test_cleaned.csv'
model_dir = root / 'models'

print('xgboost:', 'ready' if XGBClassifier is not None else 'missing')
print('tensorflow:', 'ready' if tf is not None else 'missing')


xgboost: ready
tensorflow: ready


In [2]:
df_tr = pd.read_csv(train_path)
df_te = pd.read_csv(test_path)

x = df_tr.drop(columns=[target])
y = df_tr[target].astype(int)
x_te = df_te.drop(columns=[target])
y_te = df_te[target].astype(int)

x_tr, x_va, y_tr, y_va = train_test_split(
    x, y, test_size=0.2, random_state=seed, stratify=y
)

pos_w = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))

print('train:', x_tr.shape)
print('valid:', x_va.shape)
print('test:', x_te.shape)
print('pos weight:', round(pos_w, 2))


train: (48000, 163)
valid: (12000, 163)
test: (16000, 163)
pos weight: 59.0


In [3]:
class TfBin:
    def __init__(self, in_dim, pos_w=1.0, seed=42, ep=20, bs=256):
        self.in_dim = in_dim
        self.pos_w = pos_w
        self.seed = seed
        self.ep = ep
        self.bs = bs
        self.net = None

    def _build(self):
        tf.keras.utils.set_random_seed(self.seed)
        net = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.in_dim,)),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(1, activation='sigmoid'),
        ])
        net.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='binary_crossentropy',
            metrics=[tf.keras.metrics.AUC(name='auc')],
        )
        return net

    def fit(self, x, y):
        self.net = self._build()
        cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
        self.net.fit(
            np.asarray(x),
            np.asarray(y),
            validation_split=0.1,
            epochs=self.ep,
            batch_size=self.bs,
            verbose=0,
            callbacks=cb,
            class_weight={0: 1.0, 1: self.pos_w},
        )
        return self

    def predict_proba(self, x):
        p = self.net.predict(np.asarray(x), verbose=0).reshape(-1)
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.column_stack([1 - p, p])

    def predict(self, x):
        return (self.predict_proba(x)[:, 1] >= 0.5).astype(int)


def make_lr():
    return LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear', random_state=seed)


def make_rf():
    return RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=seed,
    )


def make_et():
    return ExtraTreesClassifier(
        n_estimators=250,
        min_samples_leaf=2,
        class_weight='balanced',
        n_jobs=-1,
        random_state=seed,
    )


def make_hgb():
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=250,
        max_depth=8,
        random_state=seed,
    )


def make_dummy():
    return DummyClassifier(strategy='most_frequent')


def make_xgb():
    if XGBClassifier is None:
        return None
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
        scale_pos_weight=pos_w,
        eval_metric='logloss',
        tree_method='hist',
    )


def make_tf():
    if tf is None:
        return None
    return TfBin(in_dim=x_tr.shape[1], pos_w=pos_w, seed=seed, ep=20, bs=256)


mk = {
    'dummy': make_dummy,
    'lr': make_lr,
    'rf': make_rf,
    'et': make_et,
    'hgb': make_hgb,
    'xgb': make_xgb,
    'tf': make_tf,
}


In [4]:
def score_pos(m, x):
    if hasattr(m, 'predict_proba'):
        return m.predict_proba(x)[:, 1]
    if hasattr(m, 'decision_function'):
        z = m.decision_function(x)
        return 1.0 / (1.0 + np.exp(-z))
    return m.predict(x).astype(float)


def metric_pack(y, yp, ys):
    return {
        'acc': accuracy_score(y, yp),
        'pre': precision_score(y, yp, zero_division=0),
        'rec': recall_score(y, yp, zero_division=0),
        'f1': f1_score(y, yp, zero_division=0),
        'roc': roc_auc_score(y, ys),
        'ap': average_precision_score(y, ys),
    }


def fit_eval(make, x_fit, y_fit, x_eval, y_eval):
    m = make()
    if m is None:
        return None

    t0 = time.perf_counter()
    m.fit(x_fit, y_fit)
    dt = time.perf_counter() - t0

    ys = score_pos(m, x_eval)
    yp = (ys >= 0.5).astype(int)

    out = metric_pack(y_eval, yp, ys)
    out['sec'] = dt
    return out


def cv_eval(name, make, x_all, y_all):
    n_fold = 3 if name in {'xgb', 'tf'} else 5
    kf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=seed)

    rows = []
    for tr_idx, va_idx in kf.split(x_all, y_all):
        x1 = x_all.iloc[tr_idx]
        y1 = y_all.iloc[tr_idx]
        x2 = x_all.iloc[va_idx]
        y2 = y_all.iloc[va_idx]

        met = fit_eval(make, x1, y1, x2, y2)
        if met is None:
            return None
        rows.append(met)

    df = pd.DataFrame(rows)
    out = {f'cv_{c}': float(df[c].mean()) for c in ['acc', 'pre', 'rec', 'f1', 'roc', 'ap']}
    out['cv_ap_std'] = float(df['ap'].std(ddof=1)) if len(df) > 1 else 0.0
    return out


In [5]:
rows = []
for name, make in mk.items():
    met = fit_eval(make, x_tr, y_tr, x_va, y_va)
    if met is None:
        rows.append({'model': name, 'state': 'missing_lib'})
    else:
        rows.append({'model': name, 'state': 'ok', **met})

df_val = pd.DataFrame(rows)
df_val = df_val.sort_values(by='ap', ascending=False, na_position='last').reset_index(drop=True)
df_val


,model,state,acc,pre,rec,f1,roc,ap,sec
0,xgb,ok,0.992167,0.713710,0.885,0.790179,0.994861,0.898486,3.418773
1,hgb,ok,0.994750,0.900585,0.770,0.830189,0.994652,0.891536,2.259480
2,et,ok,0.993333,0.822581,0.765,0.792746,0.993527,0.868979,1.585706
3,rf,ok,0.992500,0.831325,0.690,0.754098,0.990272,0.861140,5.957199
4,tf,ok,0.974583,0.390397,0.935,0.550810,0.987378,0.778780,7.855373
5,lr,ok,0.972083,0.363636,0.900,0.517986,0.966008,0.731140,24.204790
6,dummy,ok,0.983333,0.000000,0.000,0.000000,0.500000,0.016667,0.003278


In [13]:
rows = []
for name, make in mk.items():
    if name not in set(df_val.loc[df_val['state'] == 'ok', 'model']):
        continue
    met = cv_eval(name, make, x, y)
    if met is None:
        continue
    rows.append({'model': name, **met})

df_cv = pd.DataFrame(rows).sort_values(by='cv_ap', ascending=False).reset_index(drop=True)
df_cv


,model,cv_acc,cv_pre,cv_rec,cv_f1,cv_roc,cv_ap,cv_ap_std
0,xgb,0.992850,0.761638,0.830993,0.794799,0.989202,0.885827,0.001266
1,hgb,0.994133,0.902479,0.727000,0.804723,0.989421,0.880929,0.015410
2,et,0.993033,0.832382,0.730000,0.777293,0.987536,0.853560,0.024412
3,rf,0.992783,0.872903,0.664000,0.753661,0.987003,0.851455,0.021637
4,tf,0.973567,0.380319,0.914996,0.536617,0.974499,0.757690,0.002715
5,lr,0.975133,0.392416,0.896000,0.545691,0.964174,0.735376,0.020809
6,dummy,0.983333,0.000000,0.000000,0.000000,0.500000,0.016667,0.000000


In [14]:
rows = []
fit = {}
for name, make in mk.items():
    if name not in set(df_val.loc[df_val['state'] == 'ok', 'model']):
        continue

    m = make()
    m.fit(x, y)
    fit[name] = m

    ys = score_pos(m, x_te)
    yp = (ys >= 0.5).astype(int)
    met = metric_pack(y_te, yp, ys)
    rows.append({'model': name, **met})

df_te = pd.DataFrame(rows).sort_values(by='ap', ascending=False).reset_index(drop=True)
df_te


,model,acc,pre,rec,f1,roc,ap
0,xgb,0.991938,0.795673,0.882667,0.836915,0.995966,0.930210
1,hgb,0.993437,0.944079,0.765333,0.845361,0.996209,0.923000
2,et,0.992687,0.898148,0.776000,0.832618,0.994644,0.900403
3,rf,0.990750,0.918819,0.664000,0.770898,0.993277,0.893625
4,tf,0.971250,0.446809,0.952000,0.608177,0.988553,0.828098
5,lr,0.975125,0.483961,0.925333,0.635531,0.979172,0.798954
6,dummy,0.976562,0.000000,0.000000,0.000000,0.500000,0.023438


In [16]:
best = df_val.loc[df_val['state'] == 'ok'].sort_values(by='ap', ascending=False).iloc[0]['model']

report = {
    'selection_metric': 'ap',
    'best_model': best,
    'validation': df_val.to_dict(orient='records'),
    'cross_validation': df_cv.to_dict(orient='records'),
    'testing': df_te.to_dict(orient='records'),
}

save = True
if save:
    model_dir.mkdir(parents=True, exist_ok=True)

    model_path = model_dir / f'best_model_{best}.joblib'
    metrics_path = model_dir / 'metrics.json'
    report_path = model_dir / 'report.txt'

    joblib.dump(fit[best], model_path)

    with metrics_path.open('w', encoding='utf-8') as f:
        json.dump(report, f, indent=2)

    with report_path.open('w', encoding='utf-8') as f:
        f.write(f'Best model: {best}\n')
        f.write('Selection metric: AP\n')
        f.write('\nValidation (AP):\n')
        for _, r in df_val[df_val['state'] == 'ok'].iterrows():
            f.write(f"- {r['model']}: {r['ap']:.4f}\n")
        f.write('\nCross-validation (AP mean):\n')
        for _, r in df_cv.iterrows():
            f.write(f"- {r['model']}: {r['cv_ap']:.4f}\n")
        f.write('\nTesting (AP):\n')
        for _, r in df_te.iterrows():
            f.write(f"- {r['model']}: {r['ap']:.4f}\n")

print('best:', best)
print('saved:', model_path)


best: xgb
saved: c:\Users\user\Downloads\BSc_Theisis\models\best_model_xgb.joblib
